# Night Vapor Dynamics

In [ ]:
import sympy as sp

# parameters
# Thrust and mass
thr_max = sp.Symbol('thr_max', real=True)  # Maximum thrust [N]

# Mass and geometry
m = sp.Symbol('m', real=True)       # Mass [kg]
XCG = sp.Symbol('XCG', real=True)   # Center of gravity (dimensionless)

# Aerodynamics
S = sp.Symbol('S', real=True)       # Wing area [m^2]
rho = sp.Symbol('rho', real=True)   # Air density [kg/m^3]
g = sp.Symbol('g', real=True)       # Gravity [m/s^2]

# Moments of inertia
Jx = sp.Symbol('Jx', real=True) # Roll moment of inertia [kg·m²]
Jy = sp.Symbol('Jy', real=True) # Pitch moment of inertia [kg·m²]
Jz = sp.Symbol('Jz', real=True) # Yaw moment of inertia [kg·m²]

# Geometry
cbar = sp.Symbol('cbar', real=True) # Mean aerodynamic chord [m]
span = sp.Symbol('span', real=True) # Wingspan [m]

# Control effectiveness (per radian)
Cm0 = sp.Symbol('Cm0', real=True)   # Zero-lift pitching moment
Cldr = sp.Symbol('Cldr', real=True) # Roll due to rudder
Cmde = sp.Symbol('Cmde', real=True) # Pitch due to elevator
Cndr = sp.Symbol('Cndr', real=True) # Yaw due to rudder
CYdr = sp.Symbol('CYdr', real=True) # Side force due to rudder

# Longitudinal stability
CL0 = sp.Symbol('CL0', real=True)       # Lift at zero AoA
CLa = sp.Symbol('CLa', real=True)       # dCL/dα [per rad]
Cma = sp.Symbol('Cma', real=True)       # dCm/dα [per rad]
Cmq = sp.Symbol('Cmq', real=True)       # Pitch damping [per rad/s]
CD0 = sp.Symbol('CD0', real=True)       # Parasite drag
CDCLS = sp.Symbol('CDCLS', real=True)   # Induced drag coefficient

# Lateral-directional stability
Cnb = sp.Symbol('Cnb', real=True)  # Yaw stiffness [per rad]
Clp = sp.Symbol('Clp', real=True)  # Roll damping [per rad/s]
Cnr = sp.Symbol('Cnr', real=True)  # Yaw damping [per rad/s]
Cnp = sp.Symbol('Cnp', real=True)  # Yaw due to roll rate
Clr = sp.Symbol('Clr', real=True)  # Roll due to yaw rate
CYb = sp.Symbol('CYb', real=True)  # Side force due to sideslip [per rad]
CYr = sp.Symbol('CYr', real=True)  # Side force due to yaw rate [per rad/s]
CYp = sp.Symbol('CYp', real=True)  # Side force due to roll rate [per rad/s]

### Parameters

In [ ]:
param_values = {
    thr_max: 0.32,
    m: 0.025,
    S: 0.025,
    rho: 1.225,
    g: 9.81,
    Jx: 1.0e-4,
    Jy: 1.0e-4,
    Jz: 1.0e-4,
    cbar: 0.09,
    span: 0.34,
    Cm0: 0.01,
    Cldr: 0.15,
    Cmde: 0.25,
    Cndr: 0.10,
    CYdr: -0.08,
    CL0: 0.6,
    CLa: 4.8,
    Cma: -0.12,
    Cmq: -0.1,
    CD0: 0.10,
    CDCLS: 0.12,
    Cnb: 0.150,
    Clp: -0.11,
    Cnr: -0.105,
    Cnp: -0.15,
    Clr: 0.10,
    CYb: -0.02,
    CYr: 0.2,
    CYp: 0.1,
}


### State Vector

In [ ]:
# state variables (13)
px, py, pz = sp.symbols('px py pz', real=True)          # position (world)
u, v, w = sp.symbols('u v w', real=True)                # velocity (body)
qw, qx, qy, qz = sp.symbols('qw qx qy qz', real=True)   # quaternion (world to body)
p, q, r = sp.symbols('p q r', real=True)                # angular velocity (body)

# state vector
X = sp.Matrix([px, py, pz, u, v, w, qw, qx, qy, qz, p, q, r])

### Control Vector

In [ ]:
# input variables (3)
thrust, elevation, rudder = sp.symbols('thrust elevation rudder', real=True)
U = sp.Matrix([thrust, elevation, rudder])

### Control Processing & Saturation

In [ ]:
# Control processing (matching cyecca implementation)
DEG2RAD = sp.pi / 180

# Control surface deflection limits (from cyecca)
max_defl = 30  # maximum rudder deflection in degrees
max_defl_elev = 24  # maximum elevator deflection in degrees

# Throttle saturation (minimum 1e-3)
throttle = sp.Piecewise(
    (1e-3, thrust < 1e-3),
    (thrust, True)
)

# Control surface deflections in radians
elev_rad = max_defl_elev * DEG2RAD * elevation  # elevator deflection
rud_rad = max_defl * DEG2RAD * rudder  # rudder deflection

# Note: Control inputs (thrust, elevation, rudder) are normalized [-1, 1]
# elev_rad and rud_rad are the actual deflections in radians

### Airspeed & Angles

In [ ]:
# Velocity saturation (matching cyecca implementation)
def saturate_velocity(vel, limit=5.0):
    """Saturate velocity components to prevent explosion"""
    return sp.Piecewise(
        (-limit, vel < -limit),
        (limit, vel > limit),
        (vel, True)
    )

# Apply saturation to velocity components (exactly like cyecca)
u_sat = saturate_velocity(u, 5.0)
v_sat = saturate_velocity(v, 5.0) 
w_sat = saturate_velocity(w, 5.0)

# Use saturated velocities for airspeed calculations
V = sp.sqrt(u_sat**2 + v_sat**2 + w_sat**2)  # airspeed

# Velocity tolerance (matching cyecca: tol_v = 1e-1 = 0.1)
tol_v = 0.1  # Aerodynamic tolerance for velocity (matching cyecca)
V_safe = sp.Max(sp.Abs(V), tol_v)  # Use absolute value like cyecca
u_safe = sp.Max(sp.Abs(u_sat), tol_v) * sp.sign(u_sat)  # Safe u velocity component

alpha = sp.atan2(-w_sat, u_safe) # angle of attack
beta = sp.asin(v_sat / V_safe)   # sideslip angle

# Angle saturation (matching cyecca: alpha -30° to +45°, beta ±30°)
DEG2RAD = sp.pi / 180
def saturate_angle(angle, min_deg, max_deg):
    """Saturate angles to prevent unrealistic values"""
    min_rad = min_deg * DEG2RAD
    max_rad = max_deg * DEG2RAD
    return sp.Piecewise(
        (min_rad, angle < min_rad),
        (max_rad, angle > max_rad),
        (angle, True)
    )

alpha = saturate_angle(alpha, -30, 45)  # -30° to +45°
beta = saturate_angle(beta, -30, 30)    # ±30°

qbar = 0.5 * rho * V_safe**2 # dynamic pressure

### Aerodynamic Coefficients

In [ ]:
# Lift
CL = CL0 + (CLa * alpha)

# Drag
CD = CD0 + (CDCLS * CL**2)

# Side force (matching cyecca implementation)
CY = -(CYb * beta) + (CYdr * rud_rad / (max_defl * DEG2RAD)) + ((span / (2 * V_safe)) * ((CYp * p) + (CYr * r)))

### Rotation Matrix (R_nb)

In [ ]:
# Rotation matrix from quaternion
def R_from_quat(qw, qx, qy, qz):
    R11 = 1 - 2*(qy**2 + qz**2)
    R12 = 2*(qx*qy - qw*qz)
    R13 = 2*(qx*qz + qw*qy)
    R21 = 2*(qx*qy + qw*qz)
    R22 = 1 - 2*(qx**2 + qz**2)
    R23 = 2*(qy*qz - qw*qx)
    R31 = 2*(qx*qz - qw*qy)
    R32 = 2*(qy*qz + qw*qx)
    R33 = 1 - 2*(qx**2 + qy**2)
    return sp.Matrix([[R11, R12, R13],
                      [R21, R22, R23],
                      [R31, R32, R33]])

# Make quaternion (wind -> body) - using R_nb convention
cos_half_alpha = sp.cos(alpha/2)
sin_half_alpha = sp.sin(alpha/2)
cos_half_beta = sp.cos(beta/2)
sin_half_beta = sp.sin(beta/2)

# q_bn (body <- wind)
qw_bn = cos_half_beta * cos_half_alpha
qx_bn = cos_half_beta * sin_half_alpha
qy_bn = sin_half_beta * cos_half_alpha
qz_bn = -sin_half_beta * sin_half_alpha

# q_nb (wind -> body) - note: this is active rotation convention
qw_nb = qw_bn
qx_nb = -qx_bn
qy_nb = -qy_bn
qz_nb = -qz_bn

# Rotation matrix (R_nb: wind -> body, active rotation)
R_nb = R_from_quat(qw_nb, qx_nb, qy_nb, qz_nb)

### Aerodynamic force

In [ ]:
Dw = qbar * S * CD  # Drag force
Lw = qbar * S * CL  # Lift force
Yw = qbar * S * CY  # Side force

# Wind frame forces (convention: X=forward, Y=right, Z=down for wind frame)
F_wind = sp.Matrix([-Dw, Yw, Lw])
F_aero_b = R_nb * F_wind  # Active rotation: wind -> body

# Thrust force (forward in body frame, using processed throttle)
T_b = sp.Matrix([thr_max * throttle, 0, 0])

# Gravity (active rotation convention)
R_wb = R_from_quat(qw, qx, qy, qz)  # world-to-body rotation
R_bw = R_wb.T  # body-to-world rotation  
W_b = R_bw * sp.Matrix([0, 0, -m*g])  # Active rotation: world -> body

# Total force in body frame
F_b = F_aero_b + T_b + W_b

### Total Moment

In [ ]:
# Moment coefficients (matching cyecca exactly - NO damping terms here)
Cl = (-1) * Cldr * rud_rad  # roll moment coefficient (basic aerodynamic only)
Cm = Cm0 + (Cma * alpha) + (Cmde * elev_rad)  # pitch moment coefficient (basic only)
Cn = (Cnb * beta) + (Cndr * rud_rad)  # yaw moment coefficient (basic only)

# Basic aerodynamic moments (without damping)
Mx_aero = qbar * S * span * Cl
My_aero = qbar * S * cbar * Cm
Mz_aero = qbar * S * span * Cn

# Damping moments (added separately, NO qbar*S multiplier, matching cyecca exactly)
Mx_damp = (Clp * (span / (2 * V_safe)) * p) + (Clr * (span / (2 * V_safe)) * r)
My_damp = (Cmq * (cbar / (2 * V_safe)) * q)
Mz_damp = (Cnp * (span / (2 * V_safe)) * p) + (Cnr * (span / (2 * V_safe)) * r)

# Total moments (aerodynamic + damping, matching cyecca exactly)
Mx_total = Mx_aero + Mx_damp
My_total = My_aero + My_damp  
Mz_total = Mz_aero + Mz_damp

M_b = sp.Matrix([Mx_total, My_total, Mz_total])

### Helper Functions

In [ ]:
def skew(ax, ay, az):
    return sp.Matrix([[0, -az,  ay],
                      [az,  0, -ax],
                      [-ay, ax,  0]])

def Omega(p, q, r):
    return sp.Matrix([
        [0,  -p, -q, -r],
        [p,   0,  r, -q],
        [q,  -r,  0,  p],
        [r,   q, -p,  0],
    ])

### Continuous-time Dynamics f(X,U)

In [ ]:
# Position kinematics (active rotation: body -> world, matching cyecca exactly)
R_wb = R_from_quat(qw, qx, qy, qz)  # world-to-body rotation matrix
R_bw = R_wb.T  # body-to-world rotation matrix
v_b = sp.Matrix([u_sat, v_sat, w_sat])  # Use saturated velocities (matching cyecca)
p_dot = R_wb * v_b  # CORRECT: R_wb matches cyecca's q_wb @ velocity_b

# Translational dynamics (using saturated velocities in cross product)
omega_b = sp.Matrix([p, q, r])
v_b_dot = (1/m) * F_b - skew(p, q, r) * v_b

# Quaternion kinematics with normalization
q_wb = sp.Matrix([qw, qx, qy, qz])
q_norm = sp.sqrt(qw**2 + qx**2 + qy**2 + qz**2)
q_wb_normalized = q_wb / q_norm
q_wb_dot = (1/2) * Omega(p, q, r) * q_wb_normalized

# Rotational dynamics
J = sp.diag(Jx, Jy, Jz)
J_w = J * omega_b
omega_b_dot = J.LUsolve(M_b - skew(p, q, r) * J_w)

# continuous-time dynamics (velocities are saturated, matching cyecca exactly)
f = sp.Matrix.vstack(p_dot, v_b_dot, q_wb_dot, omega_b_dot)

### Jacobian

In [ ]:
F = f.jacobian(X)  # Jacobian w.r.t. states
G = f.jacobian(U)  # Jacobian w.r.t. inputs

### Features Not Included (Per User Request)

The following cyecca features are excluded from this symbolic model:

1. **Stall Modeling**: cyecca includes stall condition when |alpha| >= 20° (0.3491 rad)
2. **Ground Contact Dynamics**: Landing gear forces and ground interaction
3. **Lookup Table Coefficients**: cyecca supports aerodynamic coefficient lookup tables